In [12]:
print("ok")

ok


In [13]:
import os
os.chdir("../")

In [14]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [15]:
# Extract text from PDF files
def load_pdf_files(pdf_path):
  loader = DirectoryLoader(pdf_path, glob="*.pdf", loader_cls=PyPDFLoader)
  documents = loader.load()
  return documents

In [16]:
extracted_data = load_pdf_files("data")

In [17]:
len(extracted_data)

637

In [18]:
from typing import List
from langchain.schema import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    """
    Given a list of document objects, return a new list of Document objects
    containing only 'source' in metadata and the original page_content.
    """
    minimal_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source")
        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src},
            )
        )
    return minimal_docs

In [19]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [20]:
# Split the documents into smaller chunks
def text_split(minimal_docs):
  text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap  = 20,
  )
  texts_chunks = text_splitter.split_documents(minimal_docs)
  return texts_chunks

In [21]:
texts_chunks = text_split(minimal_docs)
print(len(texts_chunks))

5859


In [22]:
from langchain.embeddings import HuggingFaceEmbeddings

def download_embeddings():
  """
  Download the HuggingFace embeddings model and save it locally.
  """
  model_name = "BAAI/bge-small-en-v1.5"
  embeddings = HuggingFaceEmbeddings(model_name=model_name)
  return embeddings

embeddings = download_embeddings()

/tmp/ipykernel_1795/891000250.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=model_name)


In [55]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [56]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY

In [57]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(api_key=pinecone_api_key)

In [69]:
from pinecone import ServerlessSpec

index_name = "medibot"

if not pc.has_index(index_name):
  pc.create_index(
    name=index_name,
    dimension=384,
    metric="cosine",
    spec=ServerlessSpec(cloud="aws", region="us-east-1")
  )

index = pc.Index(index_name)

In [70]:
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
  documents=texts_chunks,
  embedding=embeddings,
  index_name=index_name
)

In [71]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [72]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='4793c94d-b9eb-45ae-af77-986514553f86', metadata={'source': 'data/Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged with oil, dead skin\ncells, and bacteria.\nDescription\nAcne vulgaris, the medical term for common acne, is\nthe most common skin disease. It affects nearly 17 million\npeople in the United States. While acne can arise at any'),
 Document(id='254b6c75-e812-4c5e-a495-33646ddc798d', metadata={'source': 'data/Medical_book.pdf'}, page_content='of the brain. Make sure the physician knows if tetracy-\ncline is being used to treat acne or another infection.\nNancy Ross-Flanigan\nKEY TERMS\nAcne—A skin condition in which raised bumps,\npimples, and cysts form on the face, neck, shoul-\nders and upper back.\nBacteria—Tiny, one-celled forms of life tha

In [115]:
from langchain_anthropic import ChatAnthropic

chatModel = ChatAnthropic(
    model="claude-sonnet-4-6",
    api_key=ANTHROPIC_API_KEY
)

In [116]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

In [117]:
system_prompt = (
  "You are a Medical Assistant for question-answering tasks."
  "Use the following pieces of retrieved context to answer the question."
  "If you don't know the answer, just say that you don't know. DO NOT try to make up an answer."
  "Use three sentences maximum and keep the answer as concise as possible."
  "\n\n"
  "{context}"
)

prompt = ChatPromptTemplate.from_messages(
  [
    ("system", system_prompt),
    ("human", "{input}")
  ]
)

In [118]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [120]:
response = rag_chain.invoke({ "input": "what is Acromegaly and gigantism?" })
print(response["answer"])

Acromegaly is a disorder caused by the abnormal release of a particular chemical (growth hormone) from the pituitary gland in the brain, leading to increased growth in bone and soft tissue, as well as a variety of other disturbances throughout the body. Gigantism is a related condition also associated with growth hormone excess. These conditions can cause physical deformities such as enlarged feet and other skeletal changes.


In [121]:
response = rag_chain.invoke({ "input": "what is Acne?" })
print(response["answer"])

Based on the provided context, acne is a common skin disease characterized by pimples on the face, chest, and back, occurring when the pores of the skin become clogged with oil, dead skin cells, and bacteria. Its medical term is **acne vulgaris**, and it is the most common skin disease, affecting nearly 17 million people in the United States. It can also involve raised bumps, pimples, and cysts forming on the face, neck, shoulders, and upper back.
